In [2]:
import os
from collections import defaultdict
import pickle

import numpy as np

import torch
import blobfile as bf
import transformer_lens
from huggingface_hub import hf_hub_download
# from sae_lens import SparseAutoencoderDictionary
import sparse_autoencoder

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# from structured_sae.operations import (
#     low_rank_mmm,
#     kronecker_mmm,
#     sum_kronecker_mmm,
# )

In [3]:
!nvidia-smi

Sat Aug 10 17:34:46 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.14              Driver Version: 550.54.14      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:28:00.0 Off |                    0 |
| N/A   46C    P0             59W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# compute prime factorization
def prime_factors(n):
    """Returns list of prime factors of n.
    >>> prime_factors(10)
    [2, 5]
    >>> prime_factors(32)
    [2, 2, 2, 2, 2]
    """
    i = 2
    factors = []
    while i * i <= n:
        if n % i:
            i += 1
        else:
            n //= i
            factors.append(i)
    if n > 1:
        factors.append(n)
    return factors

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
torch.set_default_dtype(dtype)

In [6]:
layer_index = 7
location = "resid_post_mlp"

In [7]:
with bf.BlobFile(sparse_autoencoder.paths.v5_32k(location, layer_index), mode="rb") as f:
    state_dict = torch.load(f)
    autoencoder = sparse_autoencoder.Autoencoder.from_state_dict(state_dict)
    autoencoder.to(device)

In [8]:
W_e = autoencoder.encoder.weight.detach()
W_d = autoencoder.decoder.weight.detach()

In [9]:
W_e

tensor([[ 0.0095,  0.0370,  0.0167,  ..., -0.0111,  0.0318,  0.0216],
        [-0.0164,  0.0066, -0.0203,  ...,  0.0251, -0.0086,  0.0744],
        [ 0.0013,  0.0950,  0.0652,  ..., -0.0393,  0.1532, -0.0447],
        ...,
        [ 0.0098,  0.0186,  0.0348,  ...,  0.0154,  0.0735,  0.0014],
        [ 0.0555, -0.1026,  0.0848,  ..., -0.0417,  0.0501, -0.0143],
        [ 0.0344,  0.0175,  0.0574,  ..., -0.0532, -0.0109, -0.0415]],
       device='cuda:0')

In [10]:
# perform SVD on W_e
U, S, V = torch.svd(W_e)

In [17]:
S

tensor([143.3925,  28.4717,  19.4043,  18.0004,  17.0419,  16.4067,  16.2948,
         16.1240,  16.0347,  15.9149,  15.8261,  15.7465,  15.6844,  15.6423,
         15.5911,  15.5128,  15.4862,  15.3805,  15.3629,  15.2867,  15.2295,
         15.1888,  15.1284,  15.0839,  15.0490,  15.0283,  14.9694,  14.8899,
         14.8227,  14.8086,  14.7519,  14.7233,  14.6688,  14.6199,  14.5836,
         14.5195,  14.4582,  14.4180,  14.3672,  14.3369,  14.2763,  14.2364,
         14.2061,  14.1782,  14.1356,  14.0704,  14.0631,  14.0211,  14.0033,
         13.9845,  13.9499,  13.8656,  13.8444,  13.8057,  13.7605,  13.7192,
         13.7046,  13.6551,  13.6183,  13.5995,  13.5729,  13.5351,  13.4747,
         13.4318,  13.4196,  13.3438,  13.3185,  13.2969,  13.2703,  13.2101,
         13.2054,  13.1809,  13.1744,  13.1519,  13.1030,  13.0675,  13.0530,
         13.0178,  12.9614,  12.9085,  12.8872,  12.8676,  12.8198,  12.7810,
         12.7676,  12.7393,  12.6934,  12.6722,  12.6293,  12.57